# B3 — Training and optimization improvements

This notebook runs the pre-registered B3 optimizer/learning-rate experiment. Candidate selection uses only `unseen_layout_v1` validation dirty F1 across seeds 42, 43, and 44. Frozen test splits are evaluated only after selection. Checkpoints and reports are stored in Google Drive so the run can resume after a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_DIR = Path('/content/drive/MyDrive/ADVLSI2_B3/b3_optimization')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Persistent results: {OUTPUT_DIR}')

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPOSITORY = 'https://github.com/nocleo/ADVLSI2_Project_updated.git'
BRANCH = 'agent/b3-training-optimization'
CHECKOUT = Path(tempfile.mkdtemp(prefix='advlsi-b3-')) / 'ADVLSI2_Project_updated'

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(CHECKOUT)],
    check=True,
)
os.chdir(CHECKOUT)

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running B3.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
command = [
    sys.executable,
    'scripts/run_b3_optimization.py',
    '--python', sys.executable,
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Markdown, display
report = OUTPUT_DIR / 'README.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('No final report was written; inspect the preceding error and rerun to resume.')

In [ ]:
from google.colab import files
archive_base = '/content/ADVLSI2_B3_results'
archive = shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
files.download(archive)